# **Natural Language Processing - Assignment 3: Transformer Fine-tuning + Robustness + Limitations**

### Imports

In [ ]:
import torch

!pip install -q transformers datasets

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from pandas import DataFrame


### Loading the Dataset

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

raw = load_dataset("sh0416/ag_news")

test_set = raw["test"].shuffle(seed=13)

split = raw["train"].train_test_split(test_size=0.1, seed=13)

train_set = split["train"].shuffle(seed=13)

val_set = split["test"].shuffle(seed=13)


### Tokenization for Title and Description, Only Title, and Half of the Data

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name, max_length=128)


def tokenize_td_function(examples: dict) -> dict:
    """
    Tokenizes both the title and description of the news articles. 
    The title and description are combined into a single string for tokenization. 
    The labels are adjusted to be 0-indexed for compatibility with the model's output.

    Args:
        examples (dict): A dictionary containing the 'title', 'description', and 'label' 
        fields for a batch of examples.

    Returns:
        dict: A dictionary containing the tokenized inputs and the adjusted labels.
    """
    # Combining title and description for tokenization
    combined_texts = [
        t + " " + d for t, d in zip(examples["title"], examples["description"])
    ]
    tokenized_inputs = tokenizer(
        combined_texts, padding=True, truncation=True, max_length=128
    )
    # Add labels to the tokenized inputs and adjust them to be 0-indexed
    tokenized_inputs["labels"] = [label - 1 for label in examples["label"]]
    
    return tokenized_inputs


def tokenize_t_function(examples: dict) -> dict:
    """
    Tokenizes only the title of the news articles.
    The labels are adjusted to be 0-indexed for compatibility with the model's output.
    Args:
        examples (dict): A dictionary containing the 'title' and 'label' fields for a batch of examples.
    Returns:
        dict: A dictionary containing the tokenized inputs and the adjusted labels.
    """
    tokenized_inputs = tokenizer(
        examples["title"], padding=True, truncation=True, max_length=64
    )
    tokenized_inputs["labels"] = [label - 1 for label in examples["label"]]

    return tokenized_inputs


# Tokenizing both the title and the description
tokenized_td_train = train_set.map(tokenize_td_function, batched=True)
tokenized_td_val = val_set.map(tokenize_td_function, batched=True)
tokenized_td_test = test_set.map(tokenize_td_function, batched=True)

# Tokenizing only the title test set
tokenized_t_test = test_set.map(tokenize_t_function, batched=True)

# Creating a tokenized train set with half of the data
tokenized_td_train_50 = tokenized_td_train.select(range(60000))


Map: 100%|██████████| 100/100 [00:00<00:00, 1839.70 examples/s]


### Adding Evaluation Metric Functions

In [ ]:
def compute_metrics(eval_pred: tuple) -> dict:
    """
    Computes the accuracy and macro F1 score for the evaluation predictions.
    
    Args:
        eval_pred (tuple): A tuple containing the logits and labels from the evaluation predictions.
        
    Returns:
        dict: A dictionary containing the computed accuracy and macro F1 score.
    """
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
    }

### Loading the Pre-trained Model, Setting Up the Trainer, and Training

In [ ]:
model_td = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=4
).to(device)

# Setting up universal training arguments for fine-tuning
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_strategy="epoch",
)

# Training the model on both title and description
td_trainer = Trainer(
    model=model_td,
    args=training_args,
    train_dataset=tokenized_td_train,
    eval_dataset=tokenized_td_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

td_trainer.train()


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3347.36it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

### Training a Model on Only Half the Data

In [ ]:
model_50 = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=4
).to(device)

trainer_50 = Trainer(
    model=model_50,
    args=training_args,
    train_dataset=tokenized_td_train_50,
    eval_dataset=tokenized_td_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_50.train()

### Evaluation and Testing

In [ ]:
td_eval_results = td_trainer.evaluate(tokenized_td_test)
print(
    f"Evaluation Results on training the model on both the title and description: {td_eval_results}"
)

t_eval_results = td_trainer.evaluate(tokenized_t_test)
print(f"Evaluation Results on only the title: {t_eval_results}")

eval_results_50 = trainer_50.evaluate(tokenized_td_test)
print(f"Evaluation Results on training the model on 50% of the data: {eval_results_50}")


### Check Model Results

In [ ]:
rows = []

rows.append(
    [
        "title_only",
        t_eval_results["eval_accuracy"],
        t_eval_results["eval_macro_f1"],
        t_eval_results["eval_accuracy"],
        t_eval_results["eval_macro_f1"],
    ]
)

rows.append(
    [
        "title_description",
        td_eval_results["eval_accuracy"],
        td_eval_results["eval_macro_f1"],
        td_eval_results["eval_accuracy"],
        td_eval_results["eval_macro_f1"],
    ]
)

rows.append(
    [
        "train_50%",
        eval_results_50["eval_accuracy"],
        eval_results_50["eval_macro_f1"],
        eval_results_50["eval_accuracy"],
        eval_results_50["eval_macro_f1"],
    ]
)


### Compare Models

In [ ]:
df_compare = (
    DataFrame(
        rows,
        columns=[
            "model",
            "val_acc",
            "val_macro_f1",
            "test_acc",
            "test_macro_f1",
        ],
    )
    .sort_values(by=["val_macro_f1", "val_acc"], ascending=False)
    .reset_index(drop=True)
)

print(df_compare)

### Create Confusion Matrices

In [ ]:
def create_confusion_matrix(trainer: Trainer, tokanized_test_set: Dataset) -> None:
    """
    Creates and displays a confusion matrix for the given trainer and tokenized test set.

    Args:
        trainer (Trainer): The Trainer object used for making predictions.
        tokanized_test_set (Dataset): The tokenized test dataset to evaluate.
    
    Returns: 
        None: Displays the confusion matrix.
    """
    predictions = trainer.predict(tokanized_test_set)

    logits = predictions.predictions
    labels = predictions.label_ids

    preds = logits.argmax(axis=1)

    cm = confusion_matrix(labels, preds)
    print("Confusion Matrix:\n", cm)

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()


### Confusion Matrix for Only Title Test Set

In [ ]:
create_confusion_matrix(td_trainer, tokenized_t_test)

### Confusion Matrix for Title and Description Test Set

In [ ]:
create_confusion_matrix(td_trainer, tokenized_td_test)

### Confusion Matrix for Half of the Training Data Model

In [ ]:
create_confusion_matrix(trainer_50, tokenized_td_test)

### Error Analysis

In [ ]:
# Mapping the label indices to the corresponding class names
LABEL_NAMES_MAP = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}


def get_misclassified_df(
    trainer: Trainer, tokenized_test_set: Dataset, raw_test_set: Dataset, max_items: int = 20
) -> DataFrame:
    """
    Returns a DataFrame of up to max_items misclassified examples.
    
    Args:
        trainer (Trainer): The Trainer object used for making predictions.
        tokenized_test_set (Dataset): The tokenized test dataset to evaluate.
        raw_test_set (Dataset): The raw test dataset containing the original examples.
        max_items (int = 20): The maximum number of misclassified examples to include 
            in the DataFrame. Defaults to 20.
    
    Returns:
        DataFrame: A DataFrame containing the misclassified examples with their titles, descriptions, true labels, and predicted labels.
    """
    predictions = trainer.predict(tokenized_test_set)

    logits = predictions.predictions
    labels = predictions.label_ids

    preds = logits.argmax(axis=1)

    rows = []

    # Iterate through the predictions and labels to find misclassified examples
    for i in range(len(preds)):
        y_true = int(labels[i])
        y_pred = int(preds[i])

        if y_pred != y_true:
            ex = raw_test_set[i]

            snippet = (ex["title"] + " — " + ex["description"]).replace("\n", " ")

            rows.append(
                {
                    "title": ex["title"][:120],
                    "description": ex["description"][:200],
                    "true_label": LABEL_NAMES_MAP[y_true],
                    "pred_label": LABEL_NAMES_MAP[y_pred],
                }
            )

        # Stop at 20 misclassified examples
        if len(rows) >= max_items:
            break

    return pd.DataFrame(rows)

pd.set_option("display.max_colwidth", None)

### Misclassified Examples

In [ ]:
print("TITLE ONLY")
errors_t = get_misclassified_df(td_trainer, tokenized_t_test, test_set, max_items=20)
print(f"Showing first {len(errors_t)} misclassified examples from test set")
display(errors_t)

In [ ]:
print("\nTITLE + DESCRIPTION")
errors_td = get_misclassified_df(td_trainer, tokenized_td_test, test_set, max_items=20)
print(f"Showing first {len(errors_td)} misclassified examples from test set")
display(errors_td)